In [1]:
import torch
from transformers import RobertaTokenizer, RobertaForSequenceClassification, AdamW
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

/home/harshit21254/miniconda3/envs/EFR_Final_new/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import pandas as pd

In [3]:
# from google.colab import drive

# # Mount Google Drive
# drive.mount('/content/drive')

# Data and Model Prep

In [4]:
class CustomDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        post = self.data.iloc[idx]['post']
        label = int(self.data.iloc[idx]['offensiveYN'] * 2)  # Convert to 0, 1, or 2
        encoding = self.tokenizer(post, truncation=True, max_length=self.max_length, padding='max_length', return_tensors='pt')
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label)
        }

In [5]:
from transformers import XLNetTokenizer, XLNetForSequenceClassification

tokenizer = XLNetTokenizer.from_pretrained('xlnet-base-cased')
model = XLNetForSequenceClassification.from_pretrained('xlnet-base-cased', num_labels=3)

Some weights of XLNetForSequenceClassification were not initialized from the model checkpoint at xlnet-base-cased and are newly initialized: ['logits_proj.bias', 'logits_proj.weight', 'sequence_summary.summary.bias', 'sequence_summary.summary.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
# # Change dropout probability for the last layer of BERT
model.transformer.layer[-1].dropout.p = 0.1

# # Change dropout probability for the classifier layer
# model.classifier.dropout.p = 0.1

In [7]:
train_data = pd.read_csv("train_dataset_final.csv")
test_data = pd.read_csv("test_dataset_final.csv")
val_data = pd.read_csv("val_dataset_final.csv")


In [8]:
print(train_data.shape)
print(test_data.shape)
print(val_data.shape)

(28863, 21)
(3801, 21)
(3773, 21)


In [9]:
train_dataset = CustomDataset(train_data, tokenizer, max_length=128)
val_dataset = CustomDataset(val_data, tokenizer, max_length=128)
test_dataset = CustomDataset(test_data, tokenizer, max_length=128)

train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=4, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=4, shuffle=False)

In [10]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

XLNetForSequenceClassification(
  (transformer): XLNetModel(
    (word_embedding): Embedding(32000, 768)
    (layer): ModuleList(
      (0): XLNetLayer(
        (rel_attn): XLNetRelativeAttention(
          (layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (ff): XLNetFeedForward(
          (layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (layer_1): Linear(in_features=768, out_features=3072, bias=True)
          (layer_2): Linear(in_features=3072, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (activation_function): GELUActivation()
        )
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (1): XLNetLayer(
        (rel_attn): XLNetRelativeAttention(
          (layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (ff): XLNetFeedForward

# Training

In [20]:
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
num_epochs = 1

/home/harshit21254/miniconda3/envs/EFR_Final_new/lib/python3.10/site-packages/transformers/optimization.py:521: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [21]:
from tqdm import tqdm
import time

In [22]:
for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    start_time = time.time()

    for batch in tqdm(train_dataloader, desc=f'Epoch {epoch + 1}', leave=False):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        train_loss += loss.item()

        loss.backward()
        optimizer.step()

    train_loss /= len(train_dataloader)
    print(f'Training Loss: {train_loss}')

    model.eval()
    val_loss = 0

    with torch.no_grad():
        for batch in tqdm(val_dataloader, desc=f'Validation', leave=False):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            val_loss += loss.item()

    val_loss /= len(val_dataloader)
    print(f'Validation Loss: {val_loss}')

    end_time = time.time()  # End time of epoch
    epoch_time = end_time - start_time  # Time taken for epoch
    print(f'Epoch {epoch + 1} - Time: {epoch_time:.2f} seconds')

Training Loss: 0.3514822481786677


Validation Loss: 0.5460022955440317
Epoch 1 - Time: 1204.92 seconds


In [23]:
import pickle

model_path = "Classification_Models/XLNET_Baseline_Classifier"

with open(model_path, 'wb') as f:
    pickle.dump(model, f)

In [24]:
import pickle

model_path = "Classification_Models/XLNET_Baseline_Classifier.pkl"

with open(model_path, 'wb') as f:
    pickle.dump(model, f)

In [25]:
# test_preds = []
# test_labels = []
# for batch in test_dataloader:
#     input_ids = batch['input_ids'].to(device)
#     attention_mask = batch['attention_mask'].to(device)
#     labels = batch['labels'].to(device)
#     with torch.no_grad():
#         outputs = model(input_ids, attention_mask=attention_mask)
#         logits = outputs.logits
#     test_preds.extend(torch.argmax(logits, axis=1).cpu().numpy())
#     test_labels.extend(labels.cpu().numpy())\
# test accuracy
# test_accuracy = accuracy_score(test_labels, test_preds)

# print(f"Test Accuracy: {test_accuracy}")

from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# model.to(device)

model.eval()
predictions = []
targets = []

with torch.no_grad():
    for batch in test_dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].cpu().numpy()  # Convert labels to numpy array for sklearn metrics

        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=1)
        _, predicted_classes = torch.max(probabilities, dim=1)

        predictions.extend(predicted_classes.cpu().numpy())
        targets.extend(labels)

print('Accuracy:', accuracy_score(targets, predictions))
print('Precision:', precision_score(targets, predictions, average='weighted'))
print('Recall:', recall_score(targets, predictions, average='weighted'))
print('F1 Score:', f1_score(targets, predictions, average='weighted'))

# Print classification report
print('\nClassification Report:')
class_report = classification_report(targets, predictions)
print(class_report)

Accuracy: 0.8216258879242304
Precision: 0.807696810921134
Recall: 0.8216258879242304
F1 Score: 0.8141278992063523

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.86      0.85      1834
           1       0.17      0.11      0.14       217
           2       0.84      0.87      0.86      1750

    accuracy                           0.82      3801
   macro avg       0.62      0.61      0.62      3801
weighted avg       0.81      0.82      0.81      3801



In [26]:
report_path = "Classification_Models/XLNET_Baseline_Classifier_Report"

with open(report_path, 'wb') as f:
    pickle.dump(class_report, f)

In [27]:
report_path = "Classification_Models/XLNET_Baseline_Classifier_Report.pkl"

with open(report_path, 'wb') as f:
    pickle.dump(class_report, f)

In [28]:
model.eval()
predictionst = []
targetst = []

with torch.no_grad():
    for batch in train_dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].cpu().numpy()  # Convert labels to numpy array for sklearn metrics

        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=1)
        _, predicted_classes = torch.max(probabilities, dim=1)

        predictionst.extend(predicted_classes.cpu().numpy())
        targetst.extend(labels)

print('Accuracy:', accuracy_score(targetst, predictionst))
print('Precision:', precision_score(targetst, predictionst, average='weighted'))
print('Recall:', recall_score(targetst, predictionst, average='weighted'))
print('F1 Score:', f1_score(targetst, predictionst, average='weighted'))

# Print classification report
print('\nClassification Report:')
print(classification_report(targetst, predictionst))

Accuracy: 0.9203132037556734
Precision: 0.9115271175589881
Recall: 0.9203132037556734
F1 Score: 0.9094509803934998

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.97      0.96     15563
           1       0.72      0.33      0.45      2326
           2       0.90      0.97      0.94     10974

    accuracy                           0.92     28863
   macro avg       0.86      0.76      0.78     28863
weighted avg       0.91      0.92      0.91     28863

